# 07 — RUL Target-Cap Sensitivity Experiment

**Purpose:** diagnose the very large errors on early-life validation cases where true RUL exceeds the current 200-cycle training cap. Compare caps `125`, `175`, `200`, `250`, `350`, and **uncapped** while holding the feature pipeline, engine splits, pseudo-cutoffs, XGBoost parameters, loss (`reg:pseudohubererror`, `huber_slope=10`) and sample weighting (`row_uniform`) fixed.

**Safety rules:**
- Uses training trajectories from FD001–FD004 for labels.
- Uses `test_FD00x.txt` only to sample observed test-engine terminal ages; it does **not** load the official `RUL_FD00x.txt` files.
- Candidate selection is based on the held-out validation engines only. It does not use the existing locked-audit engines for selection and does not overwrite the current model artifact.
- Reports uncapped RUL errors overall and by true-RUL band, so the impact of label truncation is visible.

Run cells from top to bottom. This notebook writes reports only; it does not save or replace a trained model.

In [ ]:
from pathlib import Path
from datetime import datetime
import json
import shutil
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from packaging.version import Version
from xgboost import XGBRegressor

warnings.filterwarnings("ignore", category=FutureWarning)

if Version(xgb.__version__) < Version("1.7.0"):
    raise RuntimeError(f"xgboost>=1.7 is required for reg:pseudohubererror; found {xgb.__version__}.")

RANDOM_STATE = 20261026
SUBSETS = ["FD001", "FD002", "FD003", "FD004"]
COLUMN_NAMES = ["unit_nr", "time_cycles", "setting_1", "setting_2", "setting_3"] + [f"s_{i}" for i in range(1, 22)]
SENSOR_COLS = [f"s_{i}" for i in range(1, 22)]
SETTING_COLS = ["setting_1", "setting_2", "setting_3"]
WINDOW = 10
TARGET_CAP = 200  # overwritten per candidate in the experiment loop
N_REGIMES = 6
N_CUTOFFS_PER_ENGINE = 3

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"
CMAPSS_DIR = PROJECT_ROOT / "CMAPSSData"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"
for folder in (RAW_DIR, MODELS_DIR, REPORTS_DIR):
    folder.mkdir(parents=True, exist_ok=True)


def locate_file(filename):
    for path in (RAW_DIR / filename, CMAPSS_DIR / filename, PROJECT_ROOT / filename):
        if path.exists():
            return path
    raise FileNotFoundError(f"Missing {filename}; checked data/raw/, CMAPSSData/, and project root.")


def load_cmapss_file(filename):
    frame = pd.read_csv(locate_file(filename), sep=r"\s+", header=None, names=COLUMN_NAMES, engine="python")
    frame = frame.dropna(axis=1, how="all")
    if frame.shape[1] != len(COLUMN_NAMES):
        raise ValueError(f"Unexpected schema in {filename}: {frame.shape}")
    return frame


def load_training_data():
    parts = []
    for subset in SUBSETS:
        part = load_cmapss_file(f"train_{subset}.txt")
        part["subset"] = subset
        part["engine_key"] = subset + "_" + part["unit_nr"].astype(int).astype(str).str.zfill(3)
        parts.append(part)
    return pd.concat(parts, ignore_index=True)


def load_test_terminal_ages_only():
    ages = {}
    rows = []
    for subset in SUBSETS:
        part = load_cmapss_file(f"test_{subset}.txt")
        terminal = part.groupby("unit_nr")["time_cycles"].max().astype(int).to_numpy()
        ages[subset] = terminal
        rows.append({"subset": subset, "test_engines": len(terminal), "median_observed_terminal_cycle": float(np.median(terminal))})
    return ages, pd.DataFrame(rows)

train_raw = load_training_data()
observed_test_ages, test_age_summary = load_test_terminal_ages_only()
print(f"XGBoost version: {xgb.__version__}")
print(f"Training rows: {len(train_raw):,} | training engines: {train_raw['engine_key'].nunique():,}")
display(test_age_summary)
print("Official test RUL label files were not loaded by this notebook.")

## 1. Build causal rolling features and reproduce the existing engine-wise split

The split seed and per-subset split logic match Notebook 06, so the cap comparison uses the same development and validation engine sets. The locked-audit engines remain excluded from fitting and candidate selection.

In [ ]:
def engineer_base_features(frame, window=WINDOW):
    ordered = frame.sort_values(["subset", "unit_nr", "time_cycles"]).reset_index(drop=True).copy()
    required = ["subset", "unit_nr", "engine_key", "time_cycles"] + SETTING_COLS + SENSOR_COLS
    features = ordered[required].copy()
    groups = ordered.groupby(["subset", "unit_nr"], sort=False)
    for sensor in SENSOR_COLS:
        signal = groups[sensor]
        features[f"{sensor}_roll_mean_{window}"] = signal.transform(lambda s: s.rolling(window=window, min_periods=1).mean())
        features[f"{sensor}_roll_std_{window}"] = signal.transform(lambda s: s.rolling(window=window, min_periods=2).std(ddof=0)).fillna(0.0)
        features[f"{sensor}_delta_1"] = signal.diff().fillna(0.0)
    for subset in SUBSETS:
        features[f"subset_{subset}"] = (ordered["subset"] == subset).astype(np.int8)
    final_cycle = ordered.groupby("engine_key")["time_cycles"].max()
    features["RUL_true"] = final_cycle.reindex(ordered["engine_key"]).to_numpy() - ordered["time_cycles"].to_numpy()
    return features


ID_COLS = {"subset", "unit_nr", "engine_key", "RUL_true"}
base = engineer_base_features(train_raw, WINDOW)
BASE_FEATURE_COLUMNS = [c for c in base.columns if c not in ID_COLS]
assert base[BASE_FEATURE_COLUMNS].notna().all().all()
assert not ({"engine_key", "unit_nr", "RUL_true", "subset"} & set(BASE_FEATURE_COLUMNS))

DEV_KEYS, VAL_KEYS, AUDIT_KEYS = [], [], []
for i, subset in enumerate(SUBSETS):
    keys = np.sort(train_raw.loc[train_raw["subset"].eq(subset), "engine_key"].unique())
    dev_keys, remainder = train_test_split(keys, test_size=0.40, random_state=RANDOM_STATE + i)
    val_keys, audit_keys = train_test_split(remainder, test_size=0.50, random_state=RANDOM_STATE + 100 + i)
    DEV_KEYS.extend(dev_keys.tolist())
    VAL_KEYS.extend(val_keys.tolist())
    AUDIT_KEYS.extend(audit_keys.tolist())

assert set(DEV_KEYS).isdisjoint(VAL_KEYS)
assert set(DEV_KEYS).isdisjoint(AUDIT_KEYS)
assert set(VAL_KEYS).isdisjoint(AUDIT_KEYS)

split_summary = []
for subset in SUBSETS:
    all_keys = set(train_raw.loc[train_raw["subset"].eq(subset), "engine_key"].unique())
    split_summary.append({"subset": subset, "development_engines": len(all_keys & set(DEV_KEYS)),
                          "validation_engines": len(all_keys & set(VAL_KEYS)), "audit_engines": len(all_keys & set(AUDIT_KEYS))})
display(pd.DataFrame(split_summary))
print(f"Split totals: development={len(DEV_KEYS)}, validation={len(VAL_KEYS)}, audit={len(AUDIT_KEYS)}")

## 2. Sample deployment-matched pseudo-cutoffs

Pseudo-cutoffs are sampled from the observed terminal cycle distribution of test sensor trajectories, but their official RUL label files are not loaded. Validation targets come only from held-out training engines, where complete run-to-failure trajectories are available.

In [ ]:
def build_age_matched_points(features, engine_keys, age_by_subset, n_per_engine=3, seed=420):
    rng = np.random.default_rng(seed)
    allowed = set(engine_keys)
    records = []
    use = features.loc[features["engine_key"].isin(allowed)]
    for engine_key, group in use.groupby("engine_key", sort=True):
        group = group.sort_values("time_cycles")
        subset = str(group["subset"].iloc[0])
        last_cycle = int(group["time_cycles"].max())
        empirical_ages = np.asarray(age_by_subset[subset], dtype=int)
        for replicate in range(n_per_engine):
            sampled_age = int(rng.choice(empirical_ages))
            cutoff = max(1, min(sampled_age, last_cycle - 1))
            candidates = group.loc[group["time_cycles"].le(cutoff)]
            if candidates.empty:
                continue
            row = candidates.iloc[-1]
            actual_cutoff = int(row["time_cycles"])
            records.append({"base_row_idx": int(row.name), "subset": subset, "engine_key": str(engine_key),
                            "unit_nr": int(row["unit_nr"]), "cutoff_cycle": actual_cutoff,
                            "sampled_test_age": sampled_age, "true_rul_cycles": float(last_cycle - actual_cutoff),
                            "replicate": int(replicate)})
    result = pd.DataFrame(records)
    if result.empty:
        raise ValueError("No pseudo-cutoff cases were generated.")
    return result

val_points = build_age_matched_points(base, VAL_KEYS, observed_test_ages, N_CUTOFFS_PER_ENGINE, seed=RANDOM_STATE + 10)
audit_points = build_age_matched_points(base, AUDIT_KEYS, observed_test_ages, N_CUTOFFS_PER_ENGINE, seed=RANDOM_STATE + 20)
print(f"Validation cases: {len(val_points)} across {val_points.engine_key.nunique()} held-out engines")
print(f"Fresh locked audit cases: {len(audit_points)} across {audit_points.engine_key.nunique()} held-out engines")
display(val_points.groupby("subset").agg(cases=("engine_key", "size"), engines=("engine_key", "nunique"),
                                          median_true_rul=("true_rul_cycles", "median"), max_true_rul=("true_rul_cycles", "max")).reset_index())

## 3. Fit regime-aware transforms using development engines only

The regime-aware feature pipeline remains fixed across all cap candidates. No validation or audit engine is used to fit the scaler, KMeans regimes, or sensor normalization statistics.

In [ ]:
def make_stats(part, source_columns):
    stats = {}
    for col in source_columns:
        vals = pd.to_numeric(part[col], errors="coerce").to_numpy(dtype=float)
        vals = vals[np.isfinite(vals)]
        stats[col] = (float(np.mean(vals)), max(float(np.std(vals, ddof=0)), 1e-6)) if len(vals) else (0.0, 1.0)
    return stats


def fit_regime_transformer(features, fit_engine_keys, random_state=RANDOM_STATE):
    source_columns = SENSOR_COLS + [f"{s}_roll_mean_{WINDOW}" for s in SENSOR_COLS]
    fit = features.loc[features["engine_key"].isin(set(fit_engine_keys))].copy().reset_index(drop=True)
    if fit.empty:
        raise ValueError("No data supplied to fit operating-regime transformer.")
    scaler = StandardScaler().fit(fit[SETTING_COLS])
    kmeans = KMeans(n_clusters=N_REGIMES, n_init=10, random_state=random_state)
    fit["_regime"] = kmeans.fit_predict(scaler.transform(fit[SETTING_COLS])).astype(int)
    group_stats = {}
    for (subset, regime), group in fit.groupby(["subset", "_regime"], sort=True):
        group_stats[(str(subset), int(regime))] = make_stats(group, source_columns)
    subset_stats = {str(s): make_stats(g, source_columns) for s, g in fit.groupby("subset", sort=True)}
    global_stats = make_stats(fit, source_columns)
    return {"setting_scaler": scaler, "kmeans": kmeans, "group_stats": group_stats,
            "subset_stats": subset_stats, "global_stats": global_stats, "n_regimes": N_REGIMES,
            "source_columns": source_columns, "rolling_window": WINDOW,
            "method": "KMeans operating regimes plus subset/regime sensor z-scores"}


def add_regime_features(features, transformer):
    out = features.copy().reset_index(drop=True)
    regime = transformer["kmeans"].predict(transformer["setting_scaler"].transform(out[SETTING_COLS])).astype(int)
    out["_regime"] = regime
    for r in range(int(transformer["n_regimes"])):
        out[f"operating_regime_{r}"] = (regime == r).astype(np.int8)
    positions = {}
    for pos, key in enumerate(zip(out["subset"].astype(str), regime)):
        positions.setdefault((key[0], int(key[1])), []).append(pos)
    for source_col in transformer["source_columns"]:
        means, stds = np.empty(len(out), dtype=float), np.empty(len(out), dtype=float)
        for key, locs in positions.items():
            stats = transformer["group_stats"].get(key, transformer["subset_stats"].get(key[0], transformer["global_stats"]))
            mean, sd = stats.get(source_col, transformer["global_stats"][source_col])
            means[locs], stds[locs] = mean, max(float(sd), 1e-6)
        out[f"{source_col}_regime_z"] = (out[source_col].to_numpy(dtype=float) - means) / stds
    out.drop(columns=["_regime"], inplace=True)
    return out


def feature_view(fit_engine_keys):
    transformer = fit_regime_transformer(base, fit_engine_keys)
    augmented = add_regime_features(base, transformer)
    regime_columns = [f"operating_regime_{i}" for i in range(N_REGIMES)] + [f"{c}_regime_z" for c in SENSOR_COLS + [f"{s}_roll_mean_{WINDOW}" for s in SENSOR_COLS]]
    columns = BASE_FEATURE_COLUMNS + regime_columns
    if augmented[columns].isna().any().any():
        raise ValueError("Non-finite features after regime transform.")
    return augmented, columns, transformer

DEV_FEATURES, FEATURE_COLUMNS, DEV_TRANSFORMER = feature_view(DEV_KEYS)
print("Fixed feature mode: regime-aware")
print("Model features:", len(FEATURE_COLUMNS))
print("The transform above was fit on development engines only.")

## 4. Define shared metrics and the fixed XGBoost loss

All candidate configurations use the same Pseudo-Huber objective and tree parameters. The only experimental change is the target cap.

In [ ]:
def capped_target(y, cap=TARGET_CAP):
    y = np.asarray(y, dtype=float)
    return y if cap is None else np.minimum(y, float(cap))


def nasa_penalty_array(y_true, y_pred):
    error = np.asarray(y_pred, dtype=float) - np.asarray(y_true, dtype=float)
    exponent = np.where(error < 0, -error / 13.0, error / 10.0)
    return np.expm1(np.clip(exponent, 0.0, 50.0))


def score_points(points, predictions):
    scored = points[["subset", "engine_key", "unit_nr", "cutoff_cycle", "true_rul_cycles"]].copy().reset_index(drop=True)
    scored["predicted_rul_cycles"] = np.maximum(0.0, np.asarray(predictions, dtype=float))
    scored["error_pred_minus_true"] = scored["predicted_rul_cycles"] - scored["true_rul_cycles"]
    scored["absolute_error"] = scored["error_pred_minus_true"].abs()
    scored["squared_error"] = scored["error_pred_minus_true"] ** 2
    scored["nasa_penalty"] = nasa_penalty_array(scored["true_rul_cycles"], scored["predicted_rul_cycles"])
    scored["within_10_cycles"] = scored["absolute_error"] <= 10
    scored["within_20_cycles"] = scored["absolute_error"] <= 20
    scored["within_30_cycles"] = scored["absolute_error"] <= 30

    # First aggregate repeated cutoffs within each engine, then average engines equally within each subset.
    engine_metrics = scored.groupby(["subset", "engine_key"], as_index=False).agg(
        cases=("engine_key", "size"), mae=("absolute_error", "mean"),
        rmse=("squared_error", lambda x: float(np.sqrt(np.mean(x)))),
        nasa_score_per_case=("nasa_penalty", "mean"),
        late_pct=("error_pred_minus_true", lambda x: float(np.mean(x > 0) * 100)),
        mean_bias=("error_pred_minus_true", "mean"),
        within_10_pct=("within_10_cycles", lambda x: float(np.mean(x) * 100)),
        within_20_pct=("within_20_cycles", lambda x: float(np.mean(x) * 100)),
        within_30_pct=("within_30_cycles", lambda x: float(np.mean(x) * 100)),
    )
    by_subset = engine_metrics.groupby("subset", as_index=False).agg(
        engines=("engine_key", "nunique"), cases=("cases", "sum"), mae=("mae", "mean"),
        rmse=("rmse", "mean"), nasa_score_per_case=("nasa_score_per_case", "mean"),
        late_pct=("late_pct", "mean"), mean_bias=("mean_bias", "mean"),
        within_10_pct=("within_10_pct", "mean"), within_20_pct=("within_20_pct", "mean"),
        within_30_pct=("within_30_pct", "mean"),
    )
    summary = {
        "n_cases": int(len(scored)), "n_engines": int(scored["engine_key"].nunique()),
        "pooled_mae": float(scored["absolute_error"].mean()),
        "pooled_rmse": float(np.sqrt(scored["squared_error"].mean())),
        "pooled_r2": float(r2_score(scored["true_rul_cycles"], scored["predicted_rul_cycles"])),
        "macro_subset_mae": float(by_subset["mae"].mean()),
        "macro_subset_rmse": float(by_subset["rmse"].mean()),
        "macro_subset_nasa_score_per_case": float(by_subset["nasa_score_per_case"].mean()),
        "macro_subset_late_pct": float(by_subset["late_pct"].mean()),
        "macro_subset_mean_bias": float(by_subset["mean_bias"].mean()),
        "macro_subset_within_10_pct": float(by_subset["within_10_pct"].mean()),
        "macro_subset_within_20_pct": float(by_subset["within_20_pct"].mean()),
        "macro_subset_within_30_pct": float(by_subset["within_30_pct"].mean()),
    }
    return summary, by_subset, scored, engine_metrics


def sample_weights(training_frame, policy):
    if policy == "row_uniform":
        return None
    if policy != "engine_balanced":
        raise ValueError(f"Unknown sample-weight policy: {policy}")
    per_engine_rows = training_frame.groupby("engine_key")["engine_key"].transform("size").to_numpy(dtype=float)
    weights = 1.0 / np.maximum(per_engine_rows, 1.0)
    return weights / weights.mean()

FIXED_XGB_PARAMS = {
    "n_estimators": 550,
    "max_depth": 5,
    "learning_rate": 0.03,
    "subsample": 0.90,
    "colsample_bytree": 0.90,
    "min_child_weight": 5,
    "reg_lambda": 7.0,
    "tree_method": "hist",
    "n_jobs": 4,
}

LOSS_CANDIDATES = [
    {"loss_id": "squarederror", "objective": "reg:squarederror", "objective_params": {}},
    {"loss_id": "pseudohuber_slope10", "objective": "reg:pseudohubererror", "objective_params": {"huber_slope": 10.0}},
    {"loss_id": "absoluteerror", "objective": "reg:absoluteerror", "objective_params": {}},
    {"loss_id": "quantile_q035", "objective": "reg:quantileerror", "objective_params": {"quantile_alpha": 0.35}},
    {"loss_id": "quantile_q050", "objective": "reg:quantileerror", "objective_params": {"quantile_alpha": 0.50}},
]
WEIGHT_POLICIES = ["row_uniform", "engine_balanced"]

def weighted_quantile(values, quantile, weights=None):
    values = np.asarray(values, dtype=float)
    if weights is None:
        return float(np.quantile(values, quantile))
    weights = np.asarray(weights, dtype=float)
    order = np.argsort(values)
    sorted_values = values[order]
    sorted_weights = weights[order]
    cumulative = np.cumsum(sorted_weights)
    cutoff = float(quantile) * float(cumulative[-1])
    pos = min(int(np.searchsorted(cumulative, cutoff, side="left")), len(sorted_values) - 1)
    return float(sorted_values[pos])


def initial_base_score(candidate, y_fit, weights):
    """Set a loss-appropriate starting level; needed for stable robust-loss initialization."""
    objective = candidate["objective"]
    if objective == "reg:squarederror":
        return float(np.average(y_fit, weights=weights)) if weights is not None else float(np.mean(y_fit))
    if objective in {"reg:pseudohubererror", "reg:absoluteerror"}:
        return weighted_quantile(y_fit, 0.50, weights)
    if objective == "reg:quantileerror":
        q = float(candidate["objective_params"].get("quantile_alpha", 0.50))
        return weighted_quantile(y_fit, q, weights)
    raise ValueError(f"Unsupported objective in this experiment: {objective}")


def make_model(candidate, y_fit, weights=None, seed=RANDOM_STATE):
    params = dict(FIXED_XGB_PARAMS)
    params.update(candidate["objective_params"])
    params["objective"] = candidate["objective"]
    params["random_state"] = int(seed)
    # Explicit loss-appropriate intercept avoids unstable automatic initialization
    # for robust losses on RUL values measured in hundreds of cycles.
    params["base_score"] = initial_base_score(candidate, y_fit, weights)
    return XGBRegressor(**params)


def fit_candidate(candidate, fit_frame, feature_columns, fit_engine_keys, seed=RANDOM_STATE):
    mask = fit_frame["engine_key"].isin(set(fit_engine_keys))
    fit_rows = fit_frame.loc[mask]
    X_fit = fit_rows[feature_columns]
    y_fit = capped_target(fit_rows["RUL_true"].to_numpy(dtype=float), TARGET_CAP)
    w = sample_weights(fit_rows, candidate["weight_policy"])
    model = make_model(candidate, y_fit, weights=w, seed=seed)
    if w is None:
        model.fit(X_fit, y_fit)
    else:
        model.fit(X_fit, y_fit, sample_weight=w)
    return model

## 5. Train target-cap candidates on the same development engines

The experiment compares `125`, `175`, `200`, `250`, `350`, and `None` (uncapped). Model selection uses the same validation pseudo-cutoff cases for every candidate. Main reporting metrics are macro-subset MAE, macro-subset NASA penalty per case, RMSE, late rate, bias, and performance by true-RUL band.

In [ ]:
# Ensure feature matrices and validation rows are aligned to the same base-row indices.
X_validation = DEV_FEATURES.iloc[val_points["base_row_idx"].to_numpy(dtype=int)][FEATURE_COLUMNS]
CAP_CANDIDATES = [125, 175, 200, 250, 350, None]

FIXED_CANDIDATE = {
    "loss_id": "pseudohuber_slope10",
    "objective": "reg:pseudohubererror",
    "objective_params": {"huber_slope": 10.0},
    "weight_policy": "row_uniform",
}

cap_comparison_rows = []
cap_validation_predictions = []
cap_by_subset_rows = []
cap_by_band_rows = []


def label_for_cap(cap):
    return "uncapped" if cap is None else f"cap_{int(cap)}"


def score_rul_bands(scored_frame):
    tmp = scored_frame.copy()
    bins = [-np.inf, 125, 200, 300, 400, np.inf]
    labels = ["0-125", "126-200", "201-300", "301-400", ">400"]
    tmp["true_rul_band"] = pd.cut(
        tmp["true_rul_cycles"], bins=bins, labels=labels, include_lowest=True, right=True
    )
    out = tmp.groupby(["subset", "true_rul_band"], observed=False).agg(
        cases=("true_rul_cycles", "size"),
        mae=("absolute_error", "mean"),
        rmse=("squared_error", lambda x: float(np.sqrt(np.mean(x)))),
        mean_bias=("error_pred_minus_true", "mean"),
        late_pct=("error_pred_minus_true", lambda x: float(np.mean(x > 0) * 100)),
        nasa_score_per_case=("nasa_penalty", "mean"),
    ).reset_index()
    return out


for idx, cap in enumerate(CAP_CANDIDATES, start=1):
    TARGET_CAP = cap  # fit_candidate reads this global when constructing training labels
    model = fit_candidate(
        FIXED_CANDIDATE,
        DEV_FEATURES,
        FEATURE_COLUMNS,
        DEV_KEYS,
        seed=RANDOM_STATE,
    )
    predictions = np.maximum(0.0, model.predict(X_validation))
    summary, by_subset, scored, engine_metrics = score_points(val_points, predictions)
    cap_id = label_for_cap(cap)
    summary_row = {
        "candidate_id": cap_id,
        "target_cap": cap,
        "loss": FIXED_CANDIDATE["objective"],
        "huber_slope": FIXED_CANDIDATE["objective_params"]["huber_slope"],
        "weight_policy": FIXED_CANDIDATE["weight_policy"],
        "rolling_window": WINDOW,
        "feature_mode": "regime_aware",
        **summary,
    }
    cap_comparison_rows.append(summary_row)
    cap_validation_predictions.append(scored.assign(candidate_id=cap_id, target_cap=cap))
    cap_by_subset_rows.append(by_subset.assign(candidate_id=cap_id, target_cap=cap))
    cap_by_band_rows.append(score_rul_bands(scored).assign(candidate_id=cap_id, target_cap=cap))
    print(
        f"[{idx}/{len(CAP_CANDIDATES)}] {cap_id}: "
        f"macro MAE={summary['macro_subset_mae']:.2f}; "
        f"macro NASA={summary['macro_subset_nasa_score_per_case']:.3g}; "
        f"late={summary['macro_subset_late_pct']:.1f}%"
    )

# Restore the project default variable after the experiment. No production artifact is touched.
TARGET_CAP = 200
cap_comparison = pd.DataFrame(cap_comparison_rows).sort_values(
    ["macro_subset_mae", "macro_subset_nasa_score_per_case", "macro_subset_rmse"],
    ascending=True,
).reset_index(drop=True)
cap_predictions_all = pd.concat(cap_validation_predictions, ignore_index=True)
cap_by_subset_all = pd.concat(cap_by_subset_rows, ignore_index=True)
cap_by_band_all = pd.concat(cap_by_band_rows, ignore_index=True)

cap_comparison_path = REPORTS_DIR / "target_cap_validation_comparison.csv"
cap_predictions_path = REPORTS_DIR / "target_cap_validation_predictions.csv"
cap_subset_path = REPORTS_DIR / "target_cap_validation_by_subset.csv"
cap_band_path = REPORTS_DIR / "target_cap_validation_by_rul_band.csv"
cap_comparison.to_csv(cap_comparison_path, index=False)
cap_predictions_all.to_csv(cap_predictions_path, index=False)
cap_by_subset_all.to_csv(cap_subset_path, index=False)
cap_by_band_all.to_csv(cap_band_path, index=False)

display(cap_comparison[[
    "candidate_id", "target_cap", "pooled_mae", "pooled_rmse", "pooled_r2",
    "macro_subset_mae", "macro_subset_rmse", "macro_subset_nasa_score_per_case",
    "macro_subset_late_pct", "macro_subset_mean_bias", "macro_subset_within_10_pct",
    "macro_subset_within_20_pct", "macro_subset_within_30_pct"
]].round(3))
print("Saved validation reports:")
for path in [cap_comparison_path, cap_predictions_path, cap_subset_path, cap_band_path]:
    print(" -", path)
print("No model artifact was saved or overwritten.")

## 6. Read the high-RUL effect and subset differences

- `target_cap_validation_by_rul_band.csv` shows whether higher caps improve prediction for engines with true RUL above 200, 300, or 400 cycles.
- `target_cap_validation_by_subset.csv` shows whether a cap helps all four subsets or only some of them.
- `target_cap_validation_comparison.csv` is the main candidate summary.

**Selection discipline:** choose a cap using validation metrics and a predeclared priority. For this project, consider macro-subset uncapped MAE the primary metric, with macro-subset NASA score and late-prediction rate as important secondary metrics. Don't choose a cap based on official test RUL labels, and don't automatically refit the production model from this notebook.

In [ ]:
# Plot cap-vs-MAE and cap-vs-NASA diagnostics for model-selection discussion.
plot_df = cap_comparison.copy()
plot_df["cap_label"] = plot_df["target_cap"].apply(lambda x: "uncapped" if pd.isna(x) else str(int(x)))
plot_df = plot_df.sort_values("target_cap", na_position="last")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
axes[0].plot(plot_df["cap_label"], plot_df["macro_subset_mae"], marker="o")
axes[0].set_title("Target cap vs macro-subset MAE")
axes[0].set_xlabel("Training target cap (cycles)")
axes[0].set_ylabel("MAE (cycles; lower is better)")
axes[0].grid(alpha=0.25)

# NASA score can become extremely large; use a log scale to show candidates meaningfully.
score_values = np.maximum(plot_df["macro_subset_nasa_score_per_case"].to_numpy(dtype=float), 1e-12)
axes[1].plot(plot_df["cap_label"], score_values, marker="o")
axes[1].set_yscale("log")
axes[1].set_title("Target cap vs NASA asymmetric score")
axes[1].set_xlabel("Training target cap (cycles)")
axes[1].set_ylabel("Macro-subset score (log scale; lower is better)")
axes[1].grid(alpha=0.25)
plt.tight_layout()
plt.savefig(REPORTS_DIR / "target_cap_validation_diagnostics.png", dpi=160, bbox_inches="tight")
plt.show()

## Interview interpretation checklist

- A cap at 200 makes training labels above 200 indistinguishable. Cases with true RUL 300–500 may therefore be systematically underpredicted; your worst-error report showed this pattern directly.
- Uncapped training can improve high-RUL MAE but may sacrifice stability/near-failure accuracy. The validation comparison determines the trade-off.
- The NASA score is exponential, so a handful of very large errors can dominate it. Always show MAE, RMSE, bias, late-prediction rate, subset metrics, and RUL-band metrics alongside it.
- Do not claim the new model improved until it improves on a defined validation criterion. This notebook does not read official RUL label files and does not overwrite the saved model.